# Practice 47 — pandas + NumPy

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — tips

Load `sns.load_dataset('tips')`.

- Extract `time_code`: first letter of `time` (e.g. `'Dinner'` → `'D'`).
- Flag `is_weekend`: True where `day` contains `'Sat'` or `'Sun'`.
- Convert `time_code`, `day`, and `smoker` to `category`.
- `groupby(['day', 'time_code'], observed=True)['tip'].mean()` — unstack it. What shape is the result?
- Among weekend diners (`is_weekend` and `time_code == 'D'`), what is the mean tip percentage (`tip / total_bill`)?

In [ ]:
tips = sns.load_dataset('tips')

# Your code here

tips['time_code'] = tips['time'].str.extract(r'(\w{1})')
tips['is_weekend'] = tips['day'].str.contains(r'Sat|Sun')
tips[['time_code', 'day','smoker']] = tips[['time_code', 'day','smoker']].astype('category')
tips.groupby(['day','time_code'], observed=True)['tip'].mean().unstack().shape

wd = tips.loc[(tips['is_weekend'])&(tips['time_code']=='D')]
(wd['tip']/wd['total_bill']).mean()






np.float64(0.15956069379400573)

---
## Level 2 — software releases

Use the DataFrame below.

- Extract `major` and `minor` version numbers from `version` (e.g. `'v1.2.3'` → `major='1'`, `minor='2'`).
- Convert `major` to `category`.
- Use a list comprehension to collect the download columns.
- Stack the download columns into long format with a `(version, month)` index. Name the levels.
- Mean downloads per `major` version per `month` — get this as an unstacked table.

In [ ]:
releases = pd.DataFrame({
    'version':       ['v1.2.3', 'v2.0.1', 'v1.3.0', 'v2.1.0', 'v1.2.4', 'v2.0.2'],
    'platform':      ['linux', 'windows', 'mac', 'linux', 'windows', 'mac'],
    'downloads_jan': [1200, 3400,  890, 2100, 1500, 2800],
    'downloads_feb': [1350, 3200,  920, 2300, 1650, 3100],
    'downloads_mar': [1100, 3600, 1050, 2500, 1800, 3400],
})

# Your code here

releases['major'] = releases['version'].str.extract(r'v(\d)')
releases['minor'] = releases['version'].str.extract(r'v\d\.(\d)')
releases['major'] = releases['major'].astype('category')
dcl = [dc for dc in releases.columns if 'downloads_' in dc]




In [76]:
releases.set_index(['version', 'major'])[dcl].stack().rename_axis(['version', 'major', 'month'])


version  major  month        
v1.2.3   1      downloads_jan    1200
                downloads_feb    1350
                downloads_mar    1100
v2.0.1   2      downloads_jan    3400
                downloads_feb    3200
                downloads_mar    3600
v1.3.0   1      downloads_jan     890
                downloads_feb     920
                downloads_mar    1050
v2.1.0   2      downloads_jan    2100
                downloads_feb    2300
                downloads_mar    2500
v1.2.4   1      downloads_jan    1500
                downloads_feb    1650
                downloads_mar    1800
v2.0.2   2      downloads_jan    2800
                downloads_feb    3100
                downloads_mar    3400
dtype: int64

In [48]:
rd = releases[dcl].set_index(releases['version'])
rd.stack().rename_axis(['version','month']).to_frame('number')

number
version month                
v1.2.3  downloads_jan    1200
        downloads_feb    1350
        downloads_mar    1100
v2.0.1  downloads_jan    3400
        downloads_feb    3200
        downloads_mar    3600
v1.3.0  downloads_jan     890
        downloads_feb     920
        downloads_mar    1050
v2.1.0  downloads_jan    2100
        downloads_feb    2300
        downloads_mar    2500
v1.2.4  downloads_jan    1500
        downloads_feb    1650
        downloads_mar    1800
v2.0.2  downloads_jan    2800
        downloads_feb    3100
        downloads_mar    3400

In [60]:
rds = rd.stack().rename_axis(['version','month']).to_frame('number')
rds_df = rds.reset_index()
rds_df['major'] = rds_df['version'].str.extract(r'v(\d)')
rds_df.groupby(['major','month'])['number'].mean().unstack()

month,downloads_feb,downloads_jan,downloads_mar
major,,,
1,1306.666667,1196.666667,1316.666667
2,2866.666667,2766.666667,3166.666667


---
## Level 3 — support tickets

Write a `.pipe()` chain:

- **`parse(df)`** — from `ticket_id` (e.g. `'SUP-2024-Q1-001'`): extract `year`, `quarter`, `ticket_num`. Add `topic` using `np.select()` + `.str.contains()`: `'billing'` where subject contains `'billing'`, `'login'` where it contains `'login'`, default `'other'`.
- **`enrich(df)`** — convert `priority` and `region` to `category`. Add `handle_tier`: `'quick'` (≤ 45 min), `'standard'` (46–90), `'slow'` (> 90). Add `is_critical`: True where `priority == 'high'` and `topic == 'login'`.

After the chain:
- `groupby(['region', 'quarter'])['resolved'].mean()` — unstack and inspect.
- Use `.xs()` to pull out just `'AMER'`.
- Which `topic` has the highest mean `handle_min`?
- What fraction of `is_critical` tickets were resolved?
- `{region: mean_handle_min}` — dict comprehension from a groupby.

In [70]:
tickets = pd.DataFrame({
    'ticket_id':  ['SUP-2024-Q1-001', 'SUP-2024-Q1-002', 'SUP-2023-Q4-003',
                   'SUP-2024-Q2-004', 'SUP-2023-Q4-005', 'SUP-2024-Q2-006',
                   'SUP-2024-Q1-007', 'SUP-2023-Q4-008'],
    'subject':    ['login issue with SSO', 'billing error on invoice',
                   'password reset not working', 'billing charge incorrect',
                   'login failed after update', 'account access denied',
                   'billing refund request', 'login page broken'],
    'priority':   ['high', 'medium', 'high', 'low', 'high', 'medium', 'low', 'high'],
    'region':     ['AMER', 'EMEA', 'APAC', 'AMER', 'EMEA', 'APAC', 'AMER', 'EMEA'],
    'handle_min': [45, 120, 30, 90, 25, 75, 110, 40],
    'resolved':   [True, True, True, False, True, False, True, True],
})

# Your code here
def parse(df):
    df['year'] = df['ticket_id'].str.extract(r'SUP-(\d{4})')
    df['quarter'] = df['ticket_id'].str.extract(r'SUP-\d{4}-(Q\d)')
    df['ticket_num'] = df['ticket_id'].str.extract(r'SUP-\d{4}-Q\d-(\d+)$')
    conds = [df['subject'].str.contains('billing'),df['subject'].str.contains('login')]
    chos = ['billing','login']
    df['topic'] = np.select(conds, chos, default='other')
    return df

def enrich(df):
    df[['priority','region']] = df[['priority','region']].astype('category')
    df['handle_tier'] = pd.cut(df['handle_min'], bins = [0,45,90,np.inf],
                               labels=['quick','standard','slow'], right=True)
    df['is_critical'] = (df['priority'] == 'high') & (df['topic'] == 'login')
    return df 


res = tickets.copy().pipe(parse).pipe(enrich)


g = res.groupby(['region','quarter'])['resolved'].mean().unstack()


/var/folders/3r/5sttq01d46zg8zxyw17j5nbw0000gn/T/ipykernel_78195/1600568389.py:36: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = res.groupby(['region','quarter'])['resolved'].mean().unstack()


In [71]:
g.xs('AMER')

quarter
Q1    1.0
Q2    0.0
Q4    NaN
Name: AMER, dtype: float64

In [72]:
res.groupby('topic')['handle_min'].mean().idxmax()

'billing'

In [74]:
res.loc[res['is_critical'],'resolved'].mean()

np.float64(1.0)

In [75]:
gb = res.groupby('region',observed=True)['handle_min'].mean()

{r: m for r, m in zip(gb.index, gb) }

{'AMER': 81.66666666666667, 'APAC': 52.5, 'EMEA': 61.666666666666664}

Write a `.pipe()` chain:

- **`parse(df)`** — from `ticket_id` (e.g. `'SUP-2024-Q1-001'`): extract `year`, `quarter`, `ticket_num`. Add `topic` using `np.select()` + `.str.contains()`: `'billing'` where subject contains `'billing'`, `'login'` where it contains `'login'`, default `'other'`.
- **`enrich(df)`** — convert `priority` and `region` to `category`. Add `handle_tier`: `'quick'` (≤ 45 min), `'standard'` (46–90), `'slow'` (> 90). Add `is_critical`: True where `priority == 'high'` and `topic == 'login'`.

After the chain:
- `groupby(['region', 'quarter'])['resolved'].mean()` — unstack and inspect.
- Use `.xs()` to pull out just `'AMER'`.
- Which `topic` has the highest mean `handle_min`?
- What fraction of `is_critical` tickets were resolved?
- `{region: mean_handle_min}` — dict comprehension from a groupby.